# Imports

In [ ]:
%pip install faiss-cpu PyMuPDF nltk sentence-transformers

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
from nltk.tokenize import sent_tokenize
import fitz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 82.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 71.0 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


# 1️⃣ Reading text from PDF
- step 1: we will read the text from a PDF file. This is the first step in the RAG process, as we need to extract the relevant information from the document before we can analyze it. We will use a PDF reader library to extract the text from the PDF file and store it in a variable for further processing.

In [2]:
def read_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    return "\n".join(page.get_text() for page in doc)

In [3]:
pdf_txt=read_pdf(r"/kaggle/input/datasets/youssefelhelwai/resume/Youssef_Elhelw_resume.pdf")
print(pdf_txt)
print(f"chars Length: {pdf_txt.__len__()}")
words=pdf_txt.split()
print(f"words Length: {len(words)}")

Youssef Elhelw
Data Scientist
GitHub • LinkedIn
youssefelhelwai@gmail.com
+20 108 038 6734
Egypt: Cairo
AI student. Passionate about data science, machine learning, and AI engineering, with a strong focus on
solving real-world problems. Gained practical experience through projects, internships, and job simulations.
Developed solid analytical and problem-solving skills, with a growing interest in fields such as NLP, computer
vision, and generative AI.
Education
Bachelor at Computers and Artificial Intelligence at Cairo University, Giza
September 2023 - July 2027
Internships
Data Scientist at Elevvo Pathways
September 2025 - October 2025
• Developed and trained 5 distinct machine learning models, covering a wide range of ML concepts.
• Engineered ML pipelines for regression and recommendation systems, delivering strong predictive
performance.
Data Analyst at Minders, Dokki, Egypt
• Acquired practical experience in data analysis through active engagement with Minders, contributing to
real

# 2️⃣ Chunking
- step 2: after reading the text, we need to split it into smaller chunks. This is important because large documents can be difficult to process and analyze. After chunking, it will be fed into embedding model to create vector representations of the text. These vectors will be stored in a vector database for efficient retrieval during the RAG process.

In [4]:
def simple_sentence_chunking(text: str, sentences_per_chunk: int = 2):
    """
    Basic semantic chunking: group sentences together
    
    Args:
        text: Input document text
        sentences_per_chunk: Number of sentences per chunk
    
    Returns:
        List of text chunks
    """
    sentences = sent_tokenize(text)
    chunks = []
    
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = ' '.join(sentences[i:i + sentences_per_chunk])
        chunks.append(chunk)
    
    return chunks


# def simple_sentence_chunking(text, chunk_size=100, overlap=25):
#     words = text.split()
#     chunks = []
#     for i in range(0, len(words), chunk_size - overlap):
#         chunk = " ".join(words[i:i + chunk_size])
#         chunks.append(chunk)
#     return chunks

In [5]:
chunks=simple_sentence_chunking(pdf_txt)
print(f"Number of chunks: {len(chunks)}")
for i in range(len(chunks)):
    print(f"Chunk {i+1}:")
    print(chunks[i])
    print()

Number of chunks: 8
Chunk 1:
Youssef Elhelw
Data Scientist
GitHub • LinkedIn
youssefelhelwai@gmail.com
+20 108 038 6734
Egypt: Cairo
AI student. Passionate about data science, machine learning, and AI engineering, with a strong focus on
solving real-world problems.

Chunk 2:
Gained practical experience through projects, internships, and job simulations. Developed solid analytical and problem-solving skills, with a growing interest in fields such as NLP, computer
vision, and generative AI.

Chunk 3:
Education
Bachelor at Computers and Artificial Intelligence at Cairo University, Giza
September 2023 - July 2027
Internships
Data Scientist at Elevvo Pathways
September 2025 - October 2025
• Developed and trained 5 distinct machine learning models, covering a wide range of ML concepts. • Engineered ML pipelines for regression and recommendation systems, delivering strong predictive
performance.

Chunk 4:
Data Analyst at Minders, Dokki, Egypt
• Acquired practical experience in data analysis t

# 3️⃣ Embedding
- step 3: after chunking the text, we will create vector representations of the text using an embedding model. This step is crucial for the RAG process, as it allows us to represent the text in a numerical format that can be easily compared and analyzed. The embeddings will be stored in a vector database for efficient retrieval during the RAG process.

In [ ]:
def embed_chunks(chunks, model_name='BAAI/bge-m3'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)
    return model, embeddings

> Note: we returned `model` to use it when we embed the query in the next step.

In [7]:
model,embeddings=embed_chunks(chunks)
print(next(model.parameters()).device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

cuda:0


In [8]:
print(embeddings)

[[-0.04636364 -0.0254875   0.00355593 ...  0.00754692 -0.01085303
   0.06258511]
 [-0.04275871 -0.0127679   0.0212924  ... -0.00213026  0.00811953
  -0.0012423 ]
 [-0.04108207 -0.02826066  0.02535464 ...  0.00752083  0.06282386
   0.03370867]
 ...
 [-0.04689506 -0.00414947  0.0027384  ... -0.02660527  0.02787781
   0.05069801]
 [-0.03674356 -0.00218171 -0.02357551 ... -0.01335755  0.05431991
   0.01419334]
 [-0.04703044 -0.03708738 -0.0050684  ... -0.03721349  0.02045539
   0.01580487]]


# 4️⃣ Vector Database
- step 4: after creating the embeddings, we will store them in a vector database.
- we will use `faiss` as our vector database, which is a popular open-source library for efficient similarity search and clustering of dense vectors. The vector database will allow us to quickly retrieve relevant chunks of text based on their embeddings during the RAG process.

In [9]:
# def create_faiss_index(embeddings):
#     dim = embeddings.shape[1]
#     index = faiss.IndexFlatL2(dim)
#     index.add(embeddings)
#     return index

def create_faiss_index(embeddings):
    global embedding_mean
    embedding_mean = embeddings.mean(axis=0, keepdims=True)
    centered = embeddings - embedding_mean
    faiss.normalize_L2(centered)  # renormalize after centering
    
    dim = centered.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(centered)
    return index

In [10]:
index=create_faiss_index(embeddings)

In [11]:
print(index)

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x797eeb5bdbf0> >


# 5️⃣ Searching Query
- step 5: after storing the embeddings in the vector database, we will search for relevant chunks of text based on a query. This is an important step in the RAG process, as it allows us to retrieve the most relevant information from the document based on the user's query. We will use the vector database to perform a similarity search and retrieve the most relevant chunks of text based on their embeddings.

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_answering = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [13]:
def generate_text(prompt, max_new_tokens=300, do_sample=True):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(llm_answering.device)
    outputs = llm_answering.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        top_k=50 if do_sample else None,
        top_p=0.95 if do_sample else None,
        temperature=0.7 if do_sample else None,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [14]:
def expand_query(question):
    prompt = f"""Generate 3 alternative phrasings of this question, using synonyms and related terms.
Return ONLY the 3 alternatives, one per line, no numbering.
Question: {question}"""

    result = generate_text(prompt, max_new_tokens=100, do_sample=False)
    variants = [q.strip("-•* ").strip() for q in result.split("\n") if q.strip()]
    variants = list(dict.fromkeys(variants))[:3]  # dedupe, cap at 3
    return [question] + variants

In [15]:
from collections import Counter

def multi_query_retrieve(question, model, index, chunks, k_retrieve=2):
    queries = expand_query(question)
    for i, c in enumerate(queries):
        print(f"Question {i}")
        print(c)
        print("---")

    id_counts = Counter()
    for q in queries:
        q_emb = model.encode([q], convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(q_emb)
        _, indices = index.search(q_emb, k_retrieve)
        id_counts.update(indices[0].tolist())

    ranked_ids = [idx for idx, count in id_counts.most_common()]

    for idx, count in id_counts.most_common():
        preview = chunks[idx][:80].replace("\n", " ")
        print(f"chunk_id={idx} | hits={count} | {preview}...")

    return [chunks[i] for i in ranked_ids]

In [16]:
question="What School does Youssef Attend?"
top_chunks=multi_query_retrieve(question,model,index,chunks)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Question 0
What School does Youssef Attend?
---
Question 1
What Educational Institution does Youssef Attend?
---
Question 2
Where is Youssef Currently Enrolled?
---
Question 3
Which Learning Center does Youssef Attend?
---
chunk_id=0 | hits=4 | Youssef Elhelw Data Scientist GitHub • LinkedIn youssefelhelwai@gmail.com +20 10...
chunk_id=2 | hits=4 | Education Bachelor at Computers and Artificial Intelligence at Cairo University,...


In [ ]:
def prompt(question, chunks):
    context = "\n\n".join(chunks)

    return f"""               
                Use ONLY the context below to answer the question.
                
                Context:
                    {context}
                
                Question:
                    {question}
            """

In [18]:
question="how can i contact with youssef?"

top_chunks=multi_query_retrieve(question,model,index,chunks,5)

prompt1=prompt(question=question,chunks=top_chunks)

answer=generate_text(prompt1)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Question 0
how can i contact with youssef?
---
Question 1
How do I reach Youssef?
---
Question 2
How can I get in touch with Youssef?
---
Question 3
What's the best way to contact Youssef?
---
chunk_id=0 | hits=4 | Youssef Elhelw Data Scientist GitHub • LinkedIn youssefelhelwai@gmail.com +20 10...
chunk_id=7 | hits=4 | • Implemented web scraping techniques to collect additional movie data from onli...
chunk_id=4 | hits=4 | Skills Programming Skills: Python, SQL, C++, Java, OOP, FastAPI Math (Linear Alg...
chunk_id=3 | hits=4 | Data Analyst at Minders, Dokki, Egypt • Acquired practical experience in data an...
chunk_id=1 | hits=3 | Gained practical experience through projects, internships, and job simulations. ...
chunk_id=2 | hits=1 | Education Bachelor at Computers and Artificial Intelligence at Cairo University,...


In [19]:
print(answer)

You can contact Youssef Elhelw via:
- Email: `youssefelhelwai@gmail.com`
- Phone: `+20 108 038 6734`
